# Restaurant AI - Object Tracking

This notebook demonstrates person tracking using DeepSORT-style algorithm for restaurant analytics.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

from utils.detection import PersonDetector, get_video_info
from utils.tracking import DeepSORTTracker
from utils.visualization import draw_boxes

## 2. Initialize Detector and Tracker

In [ ]:
# Initialize detector and tracker
detector = PersonDetector(model_path='yolov8n.pt', confidence=0.5)
tracker = DeepSORTTracker(max_age=30, min_hits=3, iou_threshold=0.3)

print("Detector and Tracker initialized")

## 3. Process Video with Tracking

In [ ]:
# Process video with tracking
VIDEO_PATH = '../data/sample_video.mp4'
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("Video not found. Using camera instead.")
    cap = cv2.VideoCapture(0)

frame_count = 0
max_frames = 30

track_histories = {}  # Store movement paths

while frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    
    timestamp = time.time()
    
    # Detect persons
    boxes, confidences, class_ids = detector.detect(frame)
    
    # Update tracker
    tracks, track_boxes, track_ids = tracker.update(
        boxes, confidences, frame, timestamp
    )
    
    # Draw tracking results
    output = draw_boxes(frame, track_boxes, ids=track_ids, confidences=None)
    
    # Store histories
    for track in tracks:
        if track.track_id not in track_histories:
            track_histories[track.track_id] = []
        track_histories[track.track_id].append(track.center)
    
    # Draw paths
    for track_id, path in track_histories.items():
        if len(path) > 1:
            for i in range(len(path) - 1):
                cv2.line(output, path[i], path[i+1], (0, 255, 255), 2)
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
    plt.title(f'Frame {frame_count} - {len(tracks)} tracks, {len(track_histories)} total IDs')
    plt.axis('off')
    plt.show()
    
    frame_count += 1
    if frame_count >= 5:
        break

cap.release()

## 4. Real-time Tracking

In [ ]:
# Real-time tracking from camera
def run_realtime_tracking(camera_id=0, confidence=0.5):
    """Run real-time person tracking from camera."""
    cap = cv2.VideoCapture(camera_id)
    detector = PersonDetector(confidence=confidence)
    tracker = DeepSORTTracker(max_age=30, min_hits=3)
    
    print("Press 'q' to quit")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        timestamp = time.time()
        boxes, confidences, class_ids = detector.detect(frame)
        tracks, track_boxes, track_ids = tracker.update(
            boxes, confidences, frame, timestamp
        )
        
        output = draw_boxes(frame, track_boxes, ids=track_ids)
        
        cv2.putText(output, f'Tracks: {len(tracks)}', (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        cv2.imshow('Person Tracking', output)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

# Uncomment to run:
# run_realtime_tracking()

## 5. Zone-Based Tracking

In [ ]:
from utils.analytics import ZoneManager

# Define zones (example coordinates - adjust for your video)
# Assuming 640x480 video
zone_manager = ZoneManager()
zone_manager.add_zone('entrance', [(200, 400), (400, 400), (400, 450), (200, 450)], (0, 255, 255))
zone_manager.add_zone('waiting', [(100, 200), (300, 200), (300, 300), (100, 300)], (0, 255, 0))
zone_manager.add_zone('dining', [(400, 100), (600, 100), (600, 300), (400, 300)], (255, 0, 0))

print("Zones defined:", list(zone_manager.zones.keys()))

## 6. Footfall Counting

In [ ]:
from utils.analytics import FootfallCounter

# Define entry/exit line (vertical line in the middle)
footfall_counter = FootfallCounter(
    line_start=(320, 0),
    line_end=(320, 480)
)

# Process frames with footfall counting
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)

frame_count = 0
prev_positions = {}

while frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    
    timestamp = time.time()
    
    boxes, confidences, class_ids = detector.detect(frame)
    tracks, track_boxes, track_ids = tracker.update(
        boxes, confidences, frame, timestamp
    )
    
    # Update footfall counter
    for track in tracks:
        if track.track_id in prev_positions:
            result = footfall_counter.update(
                track.track_id, prev_positions[track.track_id], track.center
            )
            if result:
                print(f"Frame {frame_count}: {result} - Total: {footfall_counter.get_stats()}")
        prev_positions[track.track_id] = track.center
    
    output = draw_boxes(frame, track_boxes, ids=track_ids)
    
    # Draw counting line
    cv2.line(output, footfall_counter.line_start, footfall_counter.line_end, (0, 0, 255), 3)
    
    stats = footfall_counter.get_stats()
    cv2.putText(output, f"Entries: {stats['entries']}, Exits: {stats['exits']}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    frame_count += 1
    if frame_count >= 5:
        break

cap.release()
print(f"\nFinal counts: {footfall_counter.get_stats()}")

## Summary

This notebook covers:
- Person tracking with DeepSORT-style algorithm
- Unique ID assignment for each person
- Movement path visualization
- Zone-based tracking
- Entry/exit counting with virtual line